# 02 — Signal Cleaning & R-Peak Detection (Simulated)
Bandpass filter, R-peak detection (Pan-Tompkins), RR intervals, ectopic removal.
Uses neurokit2 simulated ECG (500Hz, 60s) while PICS downloads.

In [ ]:
import neurokit2 as nk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import butter, filtfilt

## 1. Load simulated data (replaces file loading)
Same 500Hz, 60s as pipeline. No wfdb required.

In [ ]:
ecg_raw = nk.ecg_simulate(duration=60, sampling_rate=500)
fs = 500

print(f'Sampling rate: {fs} Hz')
print(f'Samples: {len(ecg_raw)} ({len(ecg_raw)/fs:.1f} seconds)')

## 2. Bandpass filter (0.5–40 Hz)
Removes baseline wander, power line noise, high-frequency equipment noise.
4th-order Butterworth, filtfilt for zero phase shift.

In [ ]:
def bandpass_filter(signal, lowcut=0.5, highcut=40, fs=500, order=4):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, signal)

ecg_filtered = bandpass_filter(ecg_raw, fs=fs)

fig, axes = plt.subplots(2, 1, figsize=(14, 6), sharex=True)
t = np.arange(len(ecg_raw)) / fs
axes[0].plot(t, ecg_raw, color='gray', linewidth=0.6)
axes[0].set_title('Raw ECG')
axes[0].set_ylabel('Amplitude')
axes[1].plot(t, ecg_filtered, color='steelblue', linewidth=0.6)
axes[1].set_title('Filtered ECG (0.5–40 Hz)')
axes[1].set_ylabel('Amplitude')
axes[1].set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## 3. R-peak detection (Pan-Tompkins)
Detects heartbeats. Preterm infants: 120–180 bpm.

In [ ]:
_, info = nk.ecg_peaks(ecg_filtered, sampling_rate=fs)
rpeaks = info['ECG_R_Peaks']

n_peaks = len(rpeaks)
bpm = n_peaks / (len(ecg_filtered) / fs / 60) if len(ecg_filtered) > 0 else 0
print(f'Detected {n_peaks} R-peaks')
print(f'Average heart rate: {bpm:.1f} bpm')

fig, ax = plt.subplots(figsize=(12, 4))
n_show = min(5000, len(ecg_filtered))  # first 10s at 500Hz
t_10s = np.arange(n_show) / fs
ax.plot(t_10s, ecg_filtered[:n_show], color='steelblue', linewidth=0.8)
peak_mask = rpeaks < n_show
ax.scatter(rpeaks[peak_mask] / fs, ecg_filtered[rpeaks[peak_mask]], color='red', s=20, zorder=5)
ax.set_title('R-peaks (first 10 s)')
ax.set_xlabel('Time (s)')
plt.tight_layout()
plt.show()

## 4. RR intervals and ectopic filter
Compute RR in ms, remove intervals >20% from local median.

In [ ]:
def filter_ectopic_beats(rr_ms, threshold_pct=0.20):
    rr = np.array(rr_ms, dtype=float)
    median_rr = np.median(rr)
    deviation = np.abs(rr - median_rr) / median_rr
    mask = deviation <= threshold_pct
    return rr[mask], mask

rr_ms = np.diff(rpeaks) / fs * 1000
rr_clean, mask = filter_ectopic_beats(rr_ms)
removed = (~mask).sum()
print(f'RR intervals: mean={rr_clean.mean():.1f} ms, std={rr_clean.std():.1f} ms')
print(f'Removed {removed} ectopic beats ({100*removed/len(mask):.1f}%)')

## 5. Save cleaned RR intervals
Output for notebook 03 HRV extraction.

In [ ]:
import os
os.makedirs('../data/processed', exist_ok=True)

record_name = 'simulated_1'
pd.DataFrame({'rr_ms': rr_clean}).to_csv(f'../data/processed/{record_name}_rr_clean.csv', index=False)
print(f'Saved to ../data/processed/{record_name}_rr_clean.csv')

## 6. Run pipeline for all 10 simulated infants
Same cleaning steps, one CSV per "infant".

In [ ]:
os.makedirs('../data/processed', exist_ok=True)

for i in range(1, 11):
    record_name = f'simulated_{i}'
    ecg_raw = nk.ecg_simulate(duration=60, sampling_rate=500, heart_rate=120 + 6 * i)
    ecg_f = bandpass_filter(ecg_raw, fs=fs)
    _, info = nk.ecg_peaks(ecg_f, sampling_rate=fs)
    rpeaks = info['ECG_R_Peaks']
    rr = np.diff(rpeaks) / fs * 1000
    rr_clean, mask = filter_ectopic_beats(rr)
    removed_pct = (~mask).sum() / len(mask) * 100
    print(f'{record_name}: {len(rr_clean)} clean beats, {removed_pct:.1f}% removed')
    pd.DataFrame({'rr_ms': rr_clean}).to_csv(f'../data/processed/{record_name}_rr_clean.csv', index=False)